# Lazy Frame & Logical Plan 🗺️

> The relational algebra tree and the user-facing `LazyFrame` API.

In [ ]:
#| default_exp lazy_frame

In [ ]:
#| export
from typing import Any, List, Literal, Optional, Union, Dict
from dataclasses import dataclass
from mxframe.lazy_expr import Expr, col
import pyarrow as pa


## Logical Plan Nodes 🧩

These classes represent the relational operations in our query plan.

In [ ]:
#| export
class LogicalPlan:
    """Base class for all logical plan nodes."""
    def signature(self) -> tuple:
        """Return a hashable structural signature for graph caching."""
        raise NotImplementedError(f"signature() not implemented for {type(self).__name__}")

@dataclass
class Scan(LogicalPlan):
    """Represents reading data from a source (e.g., a PyArrow Table)."""
    table: pa.Table
    def signature(self) -> tuple:
        return ("Scan", tuple(self.table.column_names))

@dataclass
class Filter(LogicalPlan):
    """Represents filtering rows based on a predicate."""
    input: LogicalPlan
    predicate: Expr
    def signature(self) -> tuple:
        return ("Filter", self.input.signature(), self.predicate.signature())

@dataclass
class Project(LogicalPlan):
    """Represents selecting or computing columns."""
    input: LogicalPlan
    exprs: List[Expr]
    def signature(self) -> tuple:
        return ("Project", self.input.signature(), tuple(e.signature() for e in self.exprs))

@dataclass
class Aggregate(LogicalPlan):
    """Represents grouping and aggregating data."""
    input: LogicalPlan
    group_by: List[Expr]
    aggs: List[Expr]
    def signature(self) -> tuple:
        return ("Aggregate", self.input.signature(),
                tuple(e.signature() for e in self.group_by),
                tuple(e.signature() for e in self.aggs))

@dataclass
class Sort(LogicalPlan):
    """Represents sorting rows by one or more keys."""
    input: LogicalPlan
    by: List[Expr]
    descending: List[bool]
    def signature(self) -> tuple:
        return ("Sort", self.input.signature(),
                tuple(e.signature() for e in self.by),
                tuple(self.descending))

@dataclass
class Limit(LogicalPlan):
    """Represents taking only the first N rows."""
    input: LogicalPlan
    n: int
    def signature(self) -> tuple:
        return ("Limit", self.input.signature(), self.n)

@dataclass
class Distinct(LogicalPlan):
    """Represents removing duplicate rows (optionally on a subset of columns)."""
    input: LogicalPlan
    subset: Optional[List[str]] = None
    def signature(self) -> tuple:
        return ("Distinct", self.input.signature(),
                tuple(self.subset) if self.subset else ())


@dataclass
class Join(LogicalPlan):
    """Represents an inner join between two plan trees (first two-child node)."""
    left: LogicalPlan
    right: LogicalPlan
    left_on: List[str]
    right_on: List[str]
    how: str = "inner"
    def signature(self) -> tuple:
        return ("Join", self.left.signature(), self.right.signature(),
                tuple(self.left_on), tuple(self.right_on), self.how)


## The `LazyFrame` API 🚀

This is the main entry point for users to build queries.

In [ ]:
#| export
# Device type used by compute()
DeviceType = Literal["cpu", "gpu", "auto"]


class LazyFrame:
    """A lazy DataFrame that builds a logical plan."""
    def __init__(self, plan):
        if isinstance(plan, pa.Table):
            plan = Scan(plan)
        self.plan = plan

    def select(self, *exprs):
        """Select columns or compute new ones."""
        parsed_exprs = [col(e) if isinstance(e, str) else e for e in exprs]
        return LazyFrame(Project(self.plan, parsed_exprs))

    def filter(self, predicate):
        """Filter rows based on a condition."""
        return LazyFrame(Filter(self.plan, predicate))

    def groupby(self, *by):
        """Group data by one or more columns."""
        parsed_by = [col(b) if isinstance(b, str) else b for b in by]
        return LazyGroupBy(self, parsed_by)

    def compute(self, device: DeviceType = "cpu", verbose: bool = False) -> pa.Table:
        """Execute the logical plan and return the result.

        Args:
            device: "cpu" (default), "gpu", or "auto".
                "cpu"  — always run on CPU.
                "gpu"  — run on GPU; raises RuntimeError if no GPU is available.
                "auto" — use GPU when available and N > 100K rows, else CPU.
            verbose: if True, print compile/execute timing and cache status.
        """
        from mxframe.custom_ops import CustomOpsCompiler
        return CustomOpsCompiler(device=device).compile_and_run(self.plan, verbose=verbose)


    def sort(self, *by, descending=False):
        """Sort rows by one or more columns.

        Args:
            *by: Column names (str) or Expr objects to sort by.
            descending: bool or list of bool. If a single bool, applied to all keys.
        """
        parsed_by = [col(b) if isinstance(b, str) else b for b in by]
        if isinstance(descending, bool):
            desc_list = [descending] * len(parsed_by)
        else:
            desc_list = list(descending)
        return LazyFrame(Sort(self.plan, parsed_by, desc_list))

    def limit(self, n: int):
        """Take only the first N rows."""
        return LazyFrame(Limit(self.plan, n))

    def distinct(self, *subset):
        """Remove duplicate rows, optionally considering only a subset of columns.

        Args:
            *subset: Column names to consider for uniqueness.
                     If empty, all columns are used.
        """
        sub = list(subset) if subset else None
        return LazyFrame(Distinct(self.plan, sub))


    def join(self, other, *, on=None, left_on=None, right_on=None, how="inner"):
        """Inner join with another LazyFrame.

        Args:
            other: LazyFrame to join with.
            on: Column name(s) present in both frames (shorthand for left_on=right_on).
            left_on: Column name(s) in the left frame.
            right_on: Column name(s) in the right frame.
            how: Join type (currently only "inner" is supported).
        """
        if on is not None:
            if left_on is not None or right_on is not None:
                raise ValueError("Cannot specify both 'on' and 'left_on'/'right_on'")
            left_on = [on] if isinstance(on, str) else list(on)
            right_on = list(left_on)
        else:
            if left_on is None or right_on is None:
                raise ValueError("Must specify either 'on' or both 'left_on' and 'right_on'")
            left_on = [left_on] if isinstance(left_on, str) else list(left_on)
            right_on = [right_on] if isinstance(right_on, str) else list(right_on)
        return LazyFrame(Join(self.plan, other.plan, left_on, right_on, how))


class LazyGroupBy:
    """Intermediate object for grouping operations."""
    def __init__(self, df, by):
        self.df = df
        self.by = by

    def agg(self, *aggs) -> LazyFrame:
        """Compute aggregations for each group."""
        return LazyFrame(Aggregate(self.df.plan, self.by, list(aggs)))


## Tests 🧪

Let's verify that our `LazyFrame` builds the correct `LogicalPlan`.

In [ ]:
import pyarrow as pa
from mxframe.lazy_expr import col, lit

# Dummy data
table = pa.table({'a': [1, 2, 3], 'b': [4, 5, 6]})
lf = LazyFrame(Scan(table))

# Test select
lf_sel = lf.select(col('a') + lit(1), 'b')
assert isinstance(lf_sel.plan, Project)
assert len(lf_sel.plan.exprs) == 2
assert lf_sel.plan.exprs[1].op == 'col'

# Test filter
lf_fil = lf.filter(col('a') > lit(1))
assert isinstance(lf_fil.plan, Filter)
assert lf_fil.plan.predicate.op == 'gt'

# Test groupby
lf_agg = lf.groupby('a').agg(col('b').sum())
assert isinstance(lf_agg.plan, Aggregate)
assert len(lf_agg.plan.group_by) == 1
assert len(lf_agg.plan.aggs) == 1
assert lf_agg.plan.aggs[0].op == 'sum'

print("All tests passed! 🎉")

# ── Phase 4 Tests: Sort, Limit, Distinct ──────────────────────────────────────
from mxframe.lazy_frame import Sort, Limit, Distinct

# ── Test: .sort() builds Sort node ──
lf_sort = LazyFrame(Scan(table)).sort('a', 'b')
assert isinstance(lf_sort.plan, Sort), f'Expected Sort, got {type(lf_sort.plan)}'
assert len(lf_sort.plan.by) == 2
assert lf_sort.plan.descending == [False, False]
print('✅ .sort() builds Sort node')

# ── Test: .sort() with descending ──
lf_sort_desc = LazyFrame(Scan(table)).sort('a', descending=True)
assert lf_sort_desc.plan.descending == [True]
print('✅ .sort() descending flag works')

# ── Test: .limit() builds Limit node ──
lf_limit = LazyFrame(Scan(table)).limit(10)
assert isinstance(lf_limit.plan, Limit)
assert lf_limit.plan.n == 10
print('✅ .limit() builds Limit node')

# ── Test: .distinct() builds Distinct node ──
lf_dist = LazyFrame(Scan(table)).distinct('a')
assert isinstance(lf_dist.plan, Distinct)
assert lf_dist.plan.subset == ['a']
print('✅ .distinct() builds Distinct node')

# ── Test: .distinct() with no args uses None subset ──
lf_dist_all = LazyFrame(Scan(table)).distinct()
assert lf_dist_all.plan.subset is None
print('✅ .distinct() with no args works')

# ── Test: Chaining sort + limit ──
lf_chain = LazyFrame(Scan(table)).sort('a').limit(5)
assert isinstance(lf_chain.plan, Limit)
assert isinstance(lf_chain.plan.input, Sort)
print('✅ sort + limit chaining works')

# ── Test: Signatures are hashable ──
sig1 = lf_sort.plan.signature()
sig2 = lf_limit.plan.signature()
sig3 = lf_dist.plan.signature()
s = {sig1, sig2, sig3}
assert len(s) == 3
print('✅ Sort/Limit/Distinct signatures hashable')

print('\nAll Phase 4 plan node tests passed! 🎉')


# --- Join tests ---
left_t = pa.table({"key": [1, 2, 3], "val_l": [10, 20, 30]})
right_t = pa.table({"key": [2, 3, 4], "val_r": [200, 300, 400]})
lf = LazyFrame(left_t)
rf = LazyFrame(right_t)

# on= shorthand
jnode = lf.join(rf, on="key").plan
assert isinstance(jnode, Join)
assert jnode.left_on == ["key"] and jnode.right_on == ["key"]
assert jnode.how == "inner"

# left_on/right_on
jnode2 = lf.join(rf, left_on="key", right_on="key").plan
assert isinstance(jnode2, Join)

# multi-key
jnode3 = lf.join(rf, on=["key"]).plan
assert jnode3.left_on == ["key"]

# signature is hashable
sig = jnode.signature()
assert isinstance(sig, tuple)

# two children
assert hasattr(jnode, 'left') and hasattr(jnode, 'right')
assert isinstance(jnode.left, Scan) and isinstance(jnode.right, Scan)

# chained joins: A.join(B).join(C)
third_t = pa.table({"key": [2, 3], "val_t": [2000, 3000]})
tf = LazyFrame(third_t)
chained = lf.join(rf, on="key").join(tf, on="key").plan
assert isinstance(chained, Join)
assert isinstance(chained.left, Join)  # inner join is left child

print("✓ all join plan tests passed")


All tests passed! 🎉
